# Cuaderno de Preprocesamiento, Limpieza y Muestreo de Datos
**Proyecto:** Estimación y Auditoría de Ingresos Operativos (EAIMCS 2017–2018 - INE Bolivia)  
**Asignatura:** Machine Learning (DAT-255) - UMSA  
**Propósito:** Demostrar de forma interactiva y reproducible el pipeline completo de ingeniería de datos: ingestión cruda, auditoría de integridad, limpieza de centinelas, agregación relacional, normalización categórica, conciliación del objetivo, políticas anti-fuga (anti-leakage), estabilización de varianza y partición estratificada en cohortes de entrenamiento, calibración conformal y prueba.


## 1. Configuración del Entorno y Carga de Microdatos Crudos

Cargamos las librerías necesarias y los dos archivos fuente de la encuesta EAIMCS en `data/raw/`:
- `MOD_ANUAL_S01-07_12_general_i.csv`: Datos generales de la empresa (personal, remuneraciones, costos, activos e ingresos).
- `MOD_ANUAL_S10_materiales_i.csv`: Detalle relacional de materias primas e insumos (varias filas por empresa).


In [ ]:
import os
import sys
import json
import hashlib
from decimal import Decimal, InvalidOperation
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# Estilo gráfico corporativo
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 110

# Fijar semilla global
RANDOM_STATE_SEED = 42
np.random.seed(RANDOM_STATE_SEED)

# Localizar la raíz del proyecto
PROJECT_ROOT = Path("..").resolve() if Path("..").resolve().joinpath("data").exists() else Path(".").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print(f"Directorio raíz del proyecto: {PROJECT_ROOT}")
print(f"Ruta datos crudos: {RAW_DIR}")


In [ ]:
# Carga de los archivos crudos
gen_path = RAW_DIR / "MOD_ANUAL_S01-07_12_general_i.csv"
mat_path = RAW_DIR / "MOD_ANUAL_S10_materiales_i.csv"

# Leemos las columnas objetivo como string para evitar pérdida de centavos previa
df_gen = pd.read_csv(gen_path, low_memory=False, dtype={"S00_01_A": "string", "S05_04": "string"})
df_mat = pd.read_csv(mat_path, low_memory=False)

print(f"[OK] Módulo General cargado:    {df_gen.shape[0]:,} empresas | {df_gen.shape[1]} columnas")
print(f"[OK] Módulo Materiales cargado: {df_mat.shape[0]:,} registros de insumos | {df_mat.shape[1]} columnas")
print(f"[OK] Unicidad de ID en General: {'Correcto (100% único)' if df_gen['ID'].is_unique else 'Duplicados detectados'}")
print(f"[OK] IDs únicos en Materiales:  {df_mat['ID'].nunique():,} empresas con materias primas reportadas")


## 2. Limpieza de Inconsistencias en Materiales (Sección 10)

En la auditoría de la tabla de materias primas:
1. Se detecta una fila completamente vacía (sin ID ni valores), la cual debe excluirse sin afectar registros válidos.
2. Los valores monetarios negativos son anomalías de registro y se transforman a ausentes (`NaN`).
3. El valor `99999` representa importes altos legítimos en insumos fabriles, por lo que **no** debe forzarse a centinela nulo de forma indiscriminada sin regla documental por variable.


In [ ]:
# Diagnóstico de filas vacías
blank_rows = df_mat.isna().all(axis=1)
n_blanks = int(blank_rows.sum())
print(f"Filas 100% en blanco detectadas en Materiales: {n_blanks}")

# Filtrar filas en blanco
df_mat_clean = df_mat.loc[~blank_rows].copy()
print(f"Registros útiles tras excluir filas en blanco: {len(df_mat_clean):,}")

# Tratamiento de valores monetarios negativos
for col in ["valor_co", "valor_uti"]:
    s = pd.to_numeric(df_mat_clean[col], errors="coerce")
    negativos = (s < 0).sum()
    print(f"Columna '{col}': {negativos} valores negativos convertidos a NaN (ausentes válidos)")
    df_mat_clean[f"{col}_clean"] = s.where(s >= 0)

# Verificación del código 99999
c_99999 = (df_mat_clean[["valor_co", "valor_uti"]].isin([99999, "99999", 99999.0])).sum().sum()
print(f"Frecuencia del valor 99999 en compras/utilización: {c_99999} (se conservan como importes reales)")


## 3. Agregación Relacional de la Sección 10 a Nivel Empresa

La tabla de materiales tiene una estructura *larga* (múltiples insumos por empresa).  
Para alimentar un modelo de aprendizaje supervisado a nivel empresa, colapsamos esta información calculando:
1. `n_insumos`: Conteo de insumos y materias primas declarados por empresa.
2. `total_valor_co`: Suma total de compras de materias primas en Bs.
3. `total_valor_uti`: Suma total del valor de materias primas utilizadas en el proceso productivo.

> **Regla de Negocio:** Si una empresa no declaró materias primas, `n_insumos = 0`, pero `total_valor_co` y `total_valor_uti` permanecen como `NaN` (desconocido), evitando fabricar ceros artificiales que distorsionen el entrenamiento.


In [ ]:
# Agregación relacional por ID único de empresa
df_mat_agg = df_mat_clean.groupby("ID").agg(
    n_insumos=("ID", "size"),
    total_valor_co=("valor_co_clean", lambda s: s.sum(min_count=1)),
    total_valor_uti=("valor_uti_clean", lambda s: s.sum(min_count=1))
).reset_index()

print(f"Total empresas agregadas en Sección 10: {len(df_mat_agg):,}")
display(df_mat_agg.head(5))


## 4. Fusión 1-a-1 y Normalización Categórica (Geográfica y Sectorial)

Unimos el módulo general con los insumos agregados mediante un `Left Join` con validación estricta `one_to_one`.  
Posteriormente, estandarizamos:
- `depto`: Normalización a mayúsculas de los 9 departamentos de Bolivia a partir de `C2_01`.
- `sector_macro`: Mapeo oficial de los códigos CAEB (CIIU Rev. 4) a las 16 macroactividades económicas estándar.


In [ ]:
def map_caeb_to_sector(code) -> str:
    """Mapea el código de actividad económica CAEB (CIIU Rev. 4) al macrosector oficial."""
    try:
        c_str = str(code).strip()[:2]
        num = int(c_str)
        if 1 <= num <= 3:
            return "Agropecuario y Pesca"
        elif 5 <= num <= 9:
            return "Minería e Hidrocarburos"
        elif 10 <= num <= 33:
            return "Industria Manufacturera"
        elif 35 <= num <= 39:
            return "Electricidad, Gas y Agua"
        elif 41 <= num <= 43:
            return "Construcción"
        elif 45 <= num <= 47:
            return "Comercio Mayorista y Minorista"
        elif 49 <= num <= 53:
            return "Transporte y Almacenamiento"
        elif 55 <= num <= 56:
            return "Alojamiento y Servicios de Comida"
        elif 58 <= num <= 63:
            return "Información y Comunicaciones"
        elif 64 <= num <= 66:
            return "Intermediación Financiera"
        elif 68 <= num <= 68:
            return "Actividades Inmobiliarias"
        elif 69 <= num <= 75:
            return "Servicios Profesionales y Técnicos"
        elif 77 <= num <= 82:
            return "Servicios Administrativos y de Apoyo"
        elif 85 <= num <= 85:
            return "Educación"
        elif 86 <= num <= 88:
            return "Salud y Asistencia Social"
        else:
            return "Otras Actividades de Servicios"
    except Exception:
        return "Otras Actividades de Servicios"

# Fusión con validación de integridad 1 a 1
merged = df_gen.merge(df_mat_agg, on="ID", how="left", validate="one_to_one")
merged["has_material_record"] = merged["n_insumos"].notna()
merged["n_insumos"] = merged["n_insumos"].fillna(0)

# Estandarización de categorías
merged["depto"] = merged["C2_01"].astype(str).str.strip().str.upper()
merged["sector_macro"] = merged["actividad_pricipal_codigo_V1"].apply(map_caeb_to_sector)

print(f"Total empresas tras fusión: {len(merged):,}")
print(f"Empresas con insumos detallados (Sección 10): {merged['has_material_record'].sum():,} ({merged['has_material_record'].mean():.1%})")
print(f"Empresas sin materias primas directas:        {(~merged['has_material_record']).sum():,} (comercio/servicios)")


In [ ]:
# Visualización de la distribución territorial y sectorial de la muestra
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución por Departamento
depto_counts = merged['depto'].value_counts()
sns.barplot(x=depto_counts.values, y=depto_counts.index, hue=depto_counts.index, legend=False, ax=axes[0], palette="viridis")
axes[0].set_title("Distribución Territorial de Empresas (EAIMCS)", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Número de Empresas Encuestadas")

# Distribución por Macrosector
sector_counts = merged['sector_macro'].value_counts().head(8)
sns.barplot(x=sector_counts.values, y=sector_counts.index, hue=sector_counts.index, legend=False, ax=axes[1], palette="mako")
axes[1].set_title("Principales Macrosectores Económicos (CAEB)", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Número de Empresas")

plt.tight_layout()
plt.show()


## 5. Conciliación de la Variable Objetivo y Prevención de Fuga de Datos (*Anti-Leakage*)

1. **Conciliación de Objetivo:** Se compara `S00_01_A` con `S05_04` mediante aritmética `Decimal`. Comprobamos que las discrepancias observadas (60 empresas) son estrictamente menores o iguales a Bs 1.0 (diferencias de redondeo contable).
2. **Filtro de Inclusión:** Se conservan únicamente empresas con ingresos operativos positivos válidos ($>0$).
3. **Prevención de Data Leakage:** Se excluyen del espacio de predictores todas aquellas variables que suman directamente al ingreso o que constituyen agregados calculados a posteriori por el INE (`VBP`, `VA`, `CI`, etc.).


In [ ]:
# Función de diferencia con alta precisión Decimal
def target_differences(df: pd.DataFrame) -> pd.Series:
    def difference(row: pd.Series) -> float:
        left, right = row["S00_01_A"], row["S05_04"]
        if pd.isna(left) or pd.isna(right):
            return np.nan
        try:
            return float(abs(Decimal(str(left)) - Decimal(str(right))))
        except InvalidOperation:
            return np.nan
    return df[["S00_01_A", "S05_04"]].apply(difference, axis=1)

diffs = target_differences(merged)
discrepancias_mayores_1bs = (diffs > 1.0).sum()
discrepancias_menores_1bs = ((diffs > 0) & (diffs <= 1.0)).sum()

print(f"Discrepancias contables > Bs 1.0: {discrepancias_mayores_1bs} (Tolerancia cumplida: 0 casos)")
print(f"Discrepancias por centavos de redondeo (<= Bs 1.0): {discrepancias_menores_1bs} empresas")

# Asignar target y filtrar ingresos positivos válidos
merged["target"] = pd.to_numeric(merged["S00_01_A"], errors="coerce")
valid_mask = merged["target"].notnull() & (merged["target"] > 0)
df_clean = merged[valid_mask].copy()

print(f"Empresas válidas con target > 0: {len(df_clean):,} (100% de la muestra)")

# Auditoría de Data Leakage: lista de exclusión obligatoria
EXCLUDED_LEAKAGE_COLS = [
    "S05_01", "S05_02", "S05_03", "S05_04",
    "VPA", "PC", "ISPOINF", "VIPP", "VBP",
    "EAC", "OGO", "VUMPEEI", "CI", "VA",
    "SSB", "OPP", "PS", "R", "D",
    "target", "target_log"
]
print(f"Variables contables excluidas del entrenamiento para evitar fuga: {len(EXCLUDED_LEAKAGE_COLS)}")


## 6. Estabilización de Varianza e Ingeniería de Características

La distribución del ingreso operativo en la economía boliviana presenta una severa asimetría positiva con cola pesada tipo Pareto (pocas empresas concentran volúmenes masivos de facturación).  
Para que los algoritmos lineales y basados en árboles optimicen residuos de forma estable y no queden dominados por los mega-contribuyentes, aplicamos la transformación logarítmica:
$$y_{\text{log}} = \ln(1 + x) \quad (\texttt{log1p})$$


In [ ]:
# Predictores numéricos clave de estructura productiva
PREDICTOR_NUM_COLS = [
    "S01_05_A",   # Total personal ocupado
    "S01_03_C",   # Sueldos y salarios básicos anuales
    "S01_14",     # Otras remuneraciones y aportes
    "S02_09",     # Total energía, agua y combustibles
    "S07_09_E",   # Activos fijos totales
    "S06_06_B",   # Total inventarios finales
    "n_insumos",  # Variedad de materias primas
    "total_valor_co",   # Compras de insumos
    "total_valor_uti"   # Consumo de insumos
]

# Target logarítmico
df_clean["target_log"] = np.log1p(df_clean["target"])

# Transformación log1p en predictores numéricos preservando ausentes
for col in PREDICTOR_NUM_COLS:
    if col in df_clean.columns:
        c_num = pd.to_numeric(df_clean[col], errors="coerce")
        df_clean[col] = c_num.where(c_num >= 0)
        df_clean[f"log_{col}"] = np.log1p(df_clean[col])

print("[OK] Transformación log1p aplicada con éxito a target y predictores numéricos.")


In [ ]:
# Comparativa de la distribución del Ingreso: Escala Natural vs Escala Logarítmica
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Escala Natural
axes[0].hist(df_clean["target"] / 1e6, bins=40, color="#e74c3c", edgecolor="black", alpha=0.7)
axes[0].set_title("Ingreso Operativo en Escala Natural (Bs Millones)\nSevera Asimetría Positiva / Cola Pareto", fontsize=11, fontweight='bold')
axes[0].set_xlabel("Ingreso Operativo (Millones de Bs)")
axes[0].set_ylabel("Frecuencia de Empresas")

# Escala Logarítmica
axes[1].hist(df_clean["target_log"], bins=40, color="#2ecc71", edgecolor="black", alpha=0.7)
axes[1].set_title("Ingreso Operativo en Escala Logarítmica log1p(y)\nDistribución Cuasi-Normal Estabilizada", fontsize=11, fontweight='bold')
axes[1].set_xlabel("ln(1 + Ingreso en Bs)")
axes[1].set_ylabel("Frecuencia de Empresas")

plt.tight_layout()
plt.show()


## 7. Estrategia de Muestreo y Partición en Cohortes (60% / 20% / 20%)

Siguiendo el protocolo metodológico de regresión con cuantificación de incertidumbre:
- **Entrenamiento y Ajuste (60% - 1,891 empresas):** Para ajuste de coeficientes y búsqueda de hiperparámetros.
- **Calibración Conformal (20% - 631 empresas):** Conjunto independiente para calcular los cuantiles de no-conformidad y garantizar cobertura empírica al 90%.
- **Prueba Ciega (20% - 631 empresas):** Conjunto estrictamente reservado para evaluación final y cálculo de métricas de generalización ($R^2$, RMSE, MedAPE).

Para evitar que los tramos de ingresos queden desbalanceados, la partición se realiza mediante **muestreo estratificado por deciles de la variable objetivo** (`pd.qcut`).


In [ ]:
# Creación de estratos por deciles del target
df_clean["target_strata"] = pd.qcut(df_clean["target_log"], q=10, labels=False)

# Paso 1: Separar Entrenamiento (60%) vs Resto (40%)
df_train, df_temp = train_test_split(
    df_clean,
    test_size=0.40,
    random_state=RANDOM_STATE_SEED,
    stratify=df_clean["target_strata"]
)

# Paso 2: Dividir el 40% restante en 20% Calibración y 20% Prueba
df_calib, df_test = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=RANDOM_STATE_SEED,
    stratify=df_temp["target_strata"]
)

print("Cohortes generadas:")
print(f"- Entrenamiento (60%): {len(df_train):,} empresas ({len(df_train)/len(df_clean):.1%})")
print(f"- Calibracion (20%):   {len(df_calib):,} empresas ({len(df_calib)/len(df_clean):.1%})")
print(f"- Prueba Ciega (20%):  {len(df_test):,} empresas ({len(df_test)/len(df_clean):.1%})")
print(f"Total:                 {len(df_train) + len(df_calib) + len(df_test):,} empresas")


In [ ]:
# Verificación del balance distributivo entre los tres conjuntos
summary_df = pd.DataFrame({
    "Métrica": ["Media (Bs)", "Mediana (Bs)", "Desv. Estándar (Bs)", "Min (Bs)", "Max (Bs)"],
    "Entrenamiento": [
        f"Bs {df_train['target'].mean():,.2f}",
        f"Bs {df_train['target'].median():,.2f}",
        f"Bs {df_train['target'].std():,.2f}",
        f"Bs {df_train['target'].min():,.2f}",
        f"Bs {df_train['target'].max():,.2f}"
    ],
    "Calibración": [
        f"Bs {df_calib['target'].mean():,.2f}",
        f"Bs {df_calib['target'].median():,.2f}",
        f"Bs {df_calib['target'].std():,.2f}",
        f"Bs {df_calib['target'].min():,.2f}",
        f"Bs {df_calib['target'].max():,.2f}"
    ],
    "Prueba Ciega": [
        f"Bs {df_test['target'].mean():,.2f}",
        f"Bs {df_test['target'].median():,.2f}",
        f"Bs {df_test['target'].std():,.2f}",
        f"Bs {df_test['target'].min():,.2f}",
        f"Bs {df_test['target'].max():,.2f}"
    ]
})

display(summary_df)


## 8. Verificación de Contratos y Exportación del Dataset Procesado

Verificamos que el DataFrame procesado cumpla con todos los contratos de calidad requeridos por el sistema de producción:
1. $N = 3,153$ empresas.
2. Cero variables de fuga de datos en el conjunto de predictores.
3. Coincidencia idéntica con el archivo maestro `data/processed/dataset_procesado.csv`.


In [ ]:
# Comprobaciones de calidad (Aserciones)
assert len(df_clean) == 3153, f"Error: Se esperaban 3153 filas, pero se obtuvieron {len(df_clean)}"
assert df_clean["target"].min() > 0, "Error: Existen ingresos menores o iguales a cero"
assert not df_clean["target_log"].isna().any(), "Error: Hay valores nulos en target_log"

# Comprobación de ausencia de fuga de datos en predictores seleccionados
features_selected = [f"log_{col}" for col in PREDICTOR_NUM_COLS] + ["depto", "sector_macro"]
for leak in EXCLUDED_LEAKAGE_COLS:
    assert leak not in features_selected, f"Alerta de Fuga: {leak} detectada en los predictores"

print("[OK] Todas las aserciones de calidad y contratos de datos fueron validadas exitosamente.")

# Guardado pedagógico opcional
out_csv = PROCESSED_DIR / "dataset_procesado.csv"
if out_csv.exists():
    master_df = pd.read_csv(out_csv, low_memory=False)
    print(f"[OK] El archivo maestro existente en {out_csv} tiene {master_df.shape[0]:,} filas y {master_df.shape[1]} columnas.")
else:
    df_clean.to_csv(out_csv, index=False, encoding="utf-8")
    print(f"[OK] Dataset procesado generado y guardado en: {out_csv}")

print("""
========================================================================
¡PIPELINE DE PREPROCESAMIENTO Y MUESTREO COMPLETADO EXITOSAMENTE!
El dataset está listo para ser consumido por notebooks/comparacion_modelos.ipynb
y por el script de producción models/train.py.
========================================================================
""")
